In [0]:
df = spark.read.format("parquet")\
    .load("abfss://bronze@monarchazuredatalake.dfs.core.windows.net/orders")

df.display()


#### droping the collumn, changing the name 

In [0]:
df.schema
#for reanming
df = df.withColumnRenamed("_rescued_data","rescued_data")
# now we can drop the data - after the bronze layer - why?
df = df.drop("rescued_data")
df.display()

In [0]:
#changing the data formate - DATE format

from pyspark.sql.functions import *
## withColumn will check if the column is present then it udates it or if it is not then it creates a new column
df = df.withColumn("order_date",to_timestamp(col('order_date')))


In [0]:
# create a new column, 
df = df.withColumn("year", year(col("order_date")))
df.display()

###### Windows function

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.window import Window 

# Using col("year") explicitly to tell PySpark you mean the column, not the function
windowSpec = Window.partitionBy(col("year")).orderBy(desc(col("total_amount")))

# Creating df1 safely
df = df.withColumn("flag", dense_rank().over(windowSpec))
df.display()

###### OOPS CLASS

In [0]:
df.select("order_id", "customer_id", "product_id", "total_amount").show(5)

In [0]:
from pyspark.sql.functions import col, sum as _sum, when

df.select([
    _sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in ["order_id", "customer_id", "product_id", "total_amount", "flag"]
]).show()

df.orderBy(col("total_amount").desc()) \
  .select("order_id", "customer_id", "product_id", "total_amount", "flag") \
  .show(10)

In [0]:
spark.read.parquet("abfss://bronze@monarchazuredatalake.dfs.core.windows.net/orders").show(5)

###### Data Writing

writing into the data lake silver layer

In [0]:
df.write.format("delta")\
    .mode("overwrite")\
        .save("abfss://silver@monarchazuredatalake.dfs.core.windows.net/orders_silver")

In [0]:
%sql
select * from databricks_cata.silver.orders_silver order by total_amount desc limit 10